# Parsing samples of raw data with mi instrument
This notebook outlines how to call OOI mi.instrument modules to parse raw data delivered by OOI instruments.
For a more detailed overview see this notebook: https://github.com/friedrichknuth/ooi_local_processing/blob/master/parse_process.ipynb

In [1]:
import subprocess
import pandas as pd

from pathlib import Path
from loguru import logger

In [2]:
cwd = Path.cwd()
raw_dir = cwd / "raw_data"

In [3]:
def analyze_dat():
    analyze = 'python2 -m mi.core.instrument.playback_analysis ./raw_data'
    analyze_cmd = 'conda run -n mi-racle ' + analyze
    analyze_results = subprocess.call(analyze_cmd, shell=True)

This function summarizes metadata from your raw data files.

In [4]:
analyze_dat()

Processing 1 files in ./raw_data: 
ADCPTB104
chunky (0, 0, None, None)
ascii (0, 0, None, None)
binary (1, 96357197, datetime.datetime(2025, 10, 21, 0, 0), datetime.datetime(2025, 10, 22, 0, 0))




100%|##########| 1/1 [00:00<00:00, 168.83it/s]



Instrument classes can be found at this url: https://oceanobservatories.org/instrument-class. With this information you can 
construct a command line call. For example: if you want to parse raw data from a Seabird seaphox2 ph sensor you would call the 
`mi.instrument.seabird.seaphox2.driver` module. For a Teledyne RDI ADCP `mi.instrument.teledyne.workhorse.adcp.driver`.

In [ ]:
def parse_dat_files(refdes):

    raw_list = [file.name for file in raw_dir.glob("*.dat")]
    
    for file in raw_list:

        print(file)
        playback = f'python2 -m mi.core.instrument.playback datalog mi.instrument.teledyne.workhorse.adcp.driver \
        {refdes} log:// csv:// ./raw_data/{file}'
        playback_cmd = 'conda run -n mi-racle ' + playback
        processResults = subprocess.call(playback_cmd, shell=True)
            
        print(f"exit code: {processResults}")
        logger.info(f"Done parsing {file}")


Parsed files will outpute to the cwd along with mi driver logs.

In [6]:
# this may take awhile depending on the density of the data
parse_dat_files("CE02SHBP-LJ01D-05-ADCPTB104")

ADCPTB104_10.33.14.5_2101_20251021T0000_UTC.dat


/Users/joeduprey/repos/mi-instrument/mi/core/log.py:108: YAMLLoadWarning: calling yaml.load() without Loader=... is deprecated, as the default Loader is unsafe. Please read https://msg.pyyaml.org/load for full details.
  parsed = yaml.load(logconfig)
/Users/joeduprey/repos/mi-instrument/mi/core/instrument/wrapper.py:379: YAMLLoadWarning: calling yaml.load() without Loader=... is deprecated, as the default Loader is unsafe. Please read https://msg.pyyaml.org/load for full details.
  metadata = yaml.load(open(metadata_file))

2025-10-28 11:36:44.021 | WARNING  | __main__:parse_dat_files:14 - Done parsing ADCPTB104_10.33.14.5_2101_20251021T0000_UTC.dat


2025-10-28 11:32:17,148 INFO     mi.core.instrument.publisher Publisher: max_events: 500 publish_interval: 5
2025-10-28 11:32:17,530 INFO     mi.core.instrument.publisher Publisher: max_events: 500 publish_interval: 5
2025-10-28 11:32:17,538 INFO     mi.core.instrument.publisher Publish event: {'type': 'DRIVER_ASYNC_EVENT_STATE_CHANGE', 'value': None, 'time': 1761676337.53729}
2025-10-28 11:35:12,251 INFO     mi.core.instrument.file_publisher Writing output files...
2025-10-28 11:36:43,115 INFO     mi.core.instrument.file_publisher Done writing output files...

exit code: 0
